In [1]:
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling
from rasterio.mask import mask

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"

In [3]:
population_dir = src_dir / 'demographic 2022'
population_files_f = list(population_dir.glob('egy_f*.tif'))
population_files_m = list(population_dir.glob('egy_m*.tif'))

population_files_f.sort()
population_files_m.sort()

with rasterio.open(population_files_f[0], 'r') as src:
    nodata = src.nodata

classification_file = src_dir / "EGY_DUG_2022_GRID_L1_R2025A_v1.tif"

In [6]:
com_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
com_gdf = com_gdf.to_crs("EPSG:4326")

com_gdf_f = com_gdf.copy()
com_gdf_m = com_gdf.copy()

com_gdf_f['gender'] = 'female'
com_gdf_m['gender'] = 'male'

In [9]:
with rasterio.open(population_files_f[0]) as pop_src:
    population_shape = pop_src.shape
    transform = pop_src.transform
    crs = pop_src.crs

with rasterio.open(classification_file) as cls_src:
    classification = np.empty(population_shape, dtype=np.uint8)

    reproject(
        source=rasterio.band(cls_src, 1),
        destination=classification,
        src_transform=cls_src.transform,
        src_crs=cls_src.crs,
        dst_transform=transform,
        dst_crs=crs,
        dst_shape=population_shape,
        resampling=Resampling.nearest,
    )

for population_file in population_files_f:
    col = population_file.stem
    age_code = int(col.split("_")[2])
    with rasterio.open(population_file) as pop_src:
        population = pop_src.read(1)

        masked_pop = np.where(classification == 1, population, pop_src.nodata)

        stats = zonal_stats(
            com_gdf_f,
            masked_pop,
            affine=pop_src.transform,
            stats=["sum"],
            nodata=pop_src.nodata,
        )

    com_gdf_f[age_code] = [s["sum"] for s in stats]

for population_file in population_files_m:
    col = population_file.stem
    age_code = int(col.split("_")[2])
    with rasterio.open(population_file) as pop_src:
        population = pop_src.read(1)

        masked_pop = np.where(classification == 1, population, pop_src.nodata)

        stats = zonal_stats(
            com_gdf_m,
            masked_pop,
            affine=pop_src.transform,
            stats=["sum"],
            nodata=pop_src.nodata,
        )

    com_gdf_m[age_code] = [s["sum"] for s in stats]

/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_69841/4038873482.py:24: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  population = pop_src.read(1)
/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_69841/4038873482.py:24: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  population = pop_src.read(1)
/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_69841/4038873482.py:24: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  population = pop_src.read(1)
/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_69841/4038873482.py:24: DeprecationWarning: Setting the sha

In [10]:
com_gdf_f

,OBJECTID,Name,Shape_Leng,Shape_Area,Feddan,geometry,gender,0,1,5,...,45,50,55,60,65,70,75,80,85,90
0,1,الرشيدية_ومباشر غرب رشيد,121389.174241,3.754359e+08,89389.50,"POLYGON ((30.34442 31.38367, 30.3449 31.38464,...",female,166.152542,706.409363,942.714905,...,434.482605,353.727966,298.813049,242.739899,176.233856,113.666733,60.945129,27.218948,10.717461,2.499122
1,2,الرياح البحيرى,379980.523243,1.387385e+09,330330.00,"POLYGON ((30.49714 31.11192, 30.52423 31.12877...",female,455.005432,1934.483643,2417.553223,...,1182.377686,984.814453,858.670166,667.587280,544.712341,354.474884,196.728470,95.171234,37.487766,7.334262
2,6,الخيام,164484.927752,2.457438e+08,58510.40,"POLYGON ((32.06055 26.1862, 32.0578 26.18392, ...",female,61.188511,260.146667,367.564819,...,111.518036,86.576302,79.410530,65.625351,58.221779,32.626350,16.710918,8.581465,3.614874,0.719920
3,7,نجع حمادى الغربية,541735.552974,1.950372e+09,464374.00,"MULTIPOLYGON (((32.13347 26.10908, 32.13595 26...",female,403.760681,1716.613647,2334.558105,...,762.321960,645.818848,582.536865,530.556152,440.228607,304.986145,166.034912,72.018463,25.935791,4.928016
4,8,نجع حمادى الشرقية,385861.370984,6.143672e+08,146278.00,"POLYGON ((31.00797 27.35792, 31.01181 27.35664...",female,133.250046,566.520752,752.499695,...,232.708069,199.569122,172.371094,157.693146,125.948807,86.243027,47.949894,21.482477,8.137091,1.687006
5,9,الكلابية,334344.834772,7.479556e+08,178085.00,"POLYGON ((32.31197 26.10202, 32.31112 26.10345...",female,185.482407,706.473389,1022.231079,...,389.756653,318.597382,296.302368,269.977722,230.576965,171.050507,109.386398,64.892075,46.014259,30.974676
6,11,ترعة الصف,247397.493839,4.899687e+08,116659.00,"POLYGON ((31.29589 29.95965, 31.29888 29.95779...",female,158.538620,674.036621,877.546204,...,263.297638,221.140656,186.517456,153.753372,105.458649,70.299362,38.645424,16.862133,6.352654,1.365777
7,12,البحر الصغير,169102.077473,6.934083e+08,165097.00,"MULTIPOLYGON (((31.66716 31.20426, 31.66469 31...",female,297.061462,1262.975342,1664.896240,...,770.653320,635.757690,548.302368,451.690857,381.235870,235.719299,129.364166,66.348206,28.615200,6.436092
8,13,النجايل,89164.070500,1.617347e+08,38508.30,"POLYGON ((31.05187 30.31133, 31.05326 30.30991...",female,27.874578,118.510429,156.901382,...,59.559044,49.929466,45.093815,36.026752,28.952694,19.100607,11.186577,5.505428,2.034404,0.328440
9,14,قنال العنانية,69567.786666,6.687478e+07,15922.60,"MULTIPOLYGON (((31.77007 31.39485, 31.77001 31...",female,8.712364,37.041119,47.490917,...,27.343409,25.596369,21.652779,17.943979,11.429063,7.042558,3.772265,1.665610,0.663443,0.164242


In [33]:
demographic_df = pd.read_excel("/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx", sheet_name="rural_gender_demographic")

In [34]:
demographic_gdf_f = com_gdf_f.T.iloc[7:]
demographic_gdf_m = com_gdf_m.T.iloc[7:]

demographic_gdf = pd.concat([demographic_gdf_f, demographic_gdf_m], axis=0)

In [35]:
demographic_gdf.columns = com_gdf['OBJECTID']

In [36]:
excel_df = pd.concat([demographic_df, demographic_gdf.reset_index()], axis=1).drop(columns="index")
excel_df

,gender,Code,Min age,Max age,1,2,6,7,8,9,...,55,56,57,58,59,60,62,63,65,77
0,female,0,0,0.99,166.152542,455.005432,61.188511,403.760681,133.250046,185.482407,...,184.130768,225.004822,152.662949,860.066772,313.776489,306.587952,153.913513,222.682327,633.668396,155.191162
1,female,1,1,4.00,706.409363,1934.483643,260.146667,1716.613647,566.520752,706.473389,...,782.843262,956.621582,649.055725,3656.631836,1331.298096,1303.478271,654.373596,946.750549,2694.082275,659.80481
2,female,5,5,9.00,942.714905,2417.553223,367.564819,2334.558105,752.499695,1022.231079,...,1004.073975,1269.758301,829.196167,5006.84668,1785.66687,1756.018311,902.354553,1322.713379,3709.341064,869.626404
3,female,10,10,14.00,843.530884,2217.308838,301.612610,2045.992676,645.712219,899.347839,...,902.269409,1216.517822,705.296509,4520.147461,1612.52002,1554.21582,806.993469,1081.327026,3341.791992,740.536255
4,female,15,15,19.00,779.912292,2116.948730,236.588181,1660.364990,517.699158,752.166077,...,817.502869,994.991577,682.291016,3536.986572,1261.141357,1251.37085,639.822144,820.000305,2540.56958,632.385742
5,female,20,20,24.00,729.269897,1979.765503,170.792511,1291.625854,414.462524,623.832520,...,744.390686,834.769165,640.306091,2792.485352,984.567505,1013.013794,509.173462,781.731934,1983.347412,561.040894
6,female,25,25,29.00,674.953003,1823.707520,167.816742,1172.003662,382.501465,566.349854,...,676.872314,766.305176,571.848267,2524.393066,879.864929,895.870056,472.338593,706.279297,1833.511841,514.318604
7,female,30,30,34.00,691.011658,1759.175537,161.306030,1156.423584,365.341919,567.117188,...,727.735168,783.109253,570.674438,2411.430664,816.551392,840.785217,446.403931,834.137451,1748.859619,476.913025
8,female,35,35,39.00,646.984741,1659.781128,180.435928,1107.030273,348.028473,556.565918,...,686.334839,742.141174,528.485168,2405.919189,833.367554,844.401794,454.534546,705.550415,1783.333496,444.340454
9,female,40,40,44.00,524.088989,1361.952271,126.148636,909.647766,275.139771,434.523041,...,599.949768,635.800049,453.031616,1818.840332,623.739075,629.527039,336.525452,581.577393,1308.754517,340.873199


In [32]:
# excel_path = src_dir / "input_test_egypt.xlsx"
# write_dir = pathlib.Path("/Users/hemert/OneDrive - Stichting Deltares/Documents - International Delta Toolset/Salinity/Vietnam/")
# excel_path = write_dir / 'input.xlsx'
excel_path = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"

def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)

write_excel_file(excel_df, excel_path, sheet_name="rural_gender_demographic")